# Chương 6 — Content-based recommendation trên dữ liệu LMHT

Minh chứng **item profile → user profile → cosine similarity → top-K**. Nguồn: slide Chương 6 trang 11–20 và Precision at top 10 trang 47. Vận dụng kiến thức môn học, có phần mở rộng.

Đây là phương pháp content-based riêng, không gọi bộ xếp hạng winrate là collaborative filtering. Độ tương đồng không phải xác suất thắng. Hồ sơ chỉ dùng lịch sử trước cutoff validation.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src/07_recommender'))
from content_based import ContentBasedRecommender
report=json.loads((ROOT/'reports/content_based_evaluation.json').read_text(encoding='utf-8'))
content=ContentBasedRecommender(before=report['historical_cutoff_exclusive'])
catalog=content.catalog('mid')
print('History cutoff (exclusive):',report['historical_cutoff_exclusive'])
print('Historical games:',report['historical_games'],'| Validation games:',report['heldout_games'])


History cutoff (exclusive): 2026-04-29T08:05:12+00:00
Historical games: 1801 | Validation games: 299


## 1. Item profile (slide 12–13)

Một item là tướng tại vị trí thi đấu. Năm thuộc tính là trung bình DPM, CS/phút, ward/phút, sát thương nhận/phút và tỷ trọng vàng kiếm được. Các thuộc tính mô tả kết quả vận hành trong lịch sử; chúng phụ thuộc tuyển thủ/đội/patch và không thay thế mô tả kỹ năng tướng. Không đưa `result`, winrate hoặc điểm chuyên gia vào vector.

In [2]:
display(catalog['raw'].assign(profile_games=catalog['games']).head(12))
print('Các chiều được sử dụng:',catalog['feature_names'])


Các chiều được sử dụng: ['dpm', 'cspm', 'wpm', 'damagetakenperminute', 'earnedgoldshare']


,dpm,cspm,wpm,damagetakenperminute,earnedgoldshare,profile_games
champion,,,,,,
Ahri,658.089962,8.773465,0.308933,590.869761,0.225014,252
Akali,627.100747,8.410301,0.300335,760.165384,0.224866,148
Anivia,644.460598,8.905822,0.531984,777.698139,0.210452,49
Annie,663.946390,7.595177,0.331965,485.959545,0.200348,271
Aurelion Sol,700.066617,9.158850,0.497100,591.719300,0.231206,6
Aurora,754.172251,8.418524,0.328025,650.598260,0.221301,271
Azir,707.571133,9.281534,0.479113,589.461460,0.234978,359
Cassiopeia,619.117391,8.752032,0.491473,730.226630,0.219343,77
Corki,725.453920,9.480583,0.447253,578.088820,0.254136,30


## 2. Chuẩn hóa (Chương 2) và tướng mẫu

Min-max dùng min/max của catalog lịch sử cùng vị trí. Đặc trưng không biến thiên bị loại. Dòng thiếu phép đo không được điền giả. Tướng mẫu bị loại khỏi danh sách gợi ý của chính nó.

In [3]:
similar=content.recommend_similar('Ahri','mid',top_k=5)
display(pd.DataFrame([{'champion':r['champion'],'cosine':r['cosine_similarity'],
                       'profile_games':r['profile_games']} for r in similar['recommendations']]))
display(catalog['vectors'].loc[['Ahri']+[r['champion'] for r in similar['recommendations']]])


,champion,cosine,profile_games
0,Yone,0.954875,128
1,Aurora,0.947261,271
2,Corki,0.943750,30
3,Lissandra,0.926409,12
4,Akali,0.926307,148


,dpm,cspm,wpm,damagetakenperminute,earnedgoldshare
champion,,,,,
Ahri,0.271383,0.652486,0.174982,0.351979,0.419631
Yone,0.220362,0.845687,0.431840,0.844536,0.627339
Aurora,0.491069,0.499627,0.245599,0.464696,0.363298
Corki,0.425407,0.957013,0.686600,0.327860,0.861352
Lissandra,0.159235,0.445784,0.400394,0.390702,0.308017
Akali,0.200528,0.496086,0.143181,0.671464,0.417375


## 3. User profile (slide 16–18)

Hồ sơ tuyển thủ là trung bình có trọng số các item profiles; trọng số là lượt chọn tướng đã xảy ra. Đây là implicit preference, không phải rating được người dùng cung cấp hay bằng chứng thành thạo. Nếu user mới chưa có hồ sơ, hệ thống trả trạng thái chưa đủ dữ liệu.

In [4]:
personal=content.recommend_for_player('Faker','mid',top_k=5)
weights=pd.Series(personal['profile']['item_weights'],name='historical_pick_count')
display(weights.sort_values(ascending=False).to_frame())
display(pd.Series(personal['normalized_query_profile'],name='user_profile').to_frame())
display(pd.DataFrame([{'champion':r['champion'],'cosine':r['cosine_similarity'],
                       'profile_games':r['profile_games']} for r in personal['recommendations']]))


,historical_pick_count
Ryze,23
Orianna,22
Galio,18
Azir,18
Taliyah,14
Viktor,11
Annie,11
Sylas,10
Anivia,5
Aurora,5


,user_profile
dpm,0.288521
cspm,0.623523
wpm,0.620774
damagetakenperminute,0.437817
earnedgoldshare,0.401784


,champion,cosine,profile_games
0,Ryze,0.993813,412
1,Lissandra,0.991607,12
2,Aurelion Sol,0.987573,6
3,Azir,0.986604,359
4,Cassiopeia,0.977060,77


## 4. Kiểm tra cosine bằng phép tính trực tiếp (slide 19–20)

`cos(u,v) = dot(u,v) / (norm(u)*norm(v))`. Kết quả khớp sklearn; không nhân thành “% thắng”.

In [5]:
first=personal['recommendations'][0]
u=np.array([personal['normalized_query_profile'][f] for f in catalog['feature_names']])
v=catalog['vectors'].loc[first['champion']].to_numpy()
manual=float(np.dot(u,v)/(np.linalg.norm(u)*np.linalg.norm(v)))
assert np.isclose(manual,first['cosine_similarity'])
print('Direct cosine:',manual,'| Returned cosine:',first['cosine_similarity'])
assert personal['probability'] is None and personal['causal_uplift'] is None
print('Cold start:',content.recommend_for_player('__new_player__','mid')['status'])


Direct cosine: 0.993813091058138 | Returned cosine: 0.993813091058138
Cold start: insufficient_profile


## 5. Đánh giá theo bài học và mở rộng

Precision@10 có trong slide 47. Recall/MRR, cutoff theo thời gian và CI bootstrap là phần mở rộng. Đây là validation holdout, không phải kết quả final test mới. Một nhãn quan sát/truy vấn khiến Precision@10 tối đa 0.1; không thể suy ra chín tướng khác không phù hợp. Bài toán hồi cứu biết chín tướng còn lại và không tái hiện thứ tự draft thật.

In [6]:
display(pd.DataFrame(report['overall']).T)
display(pd.DataFrame(report['warm_user_cohort']).T)
print('User profile coverage:',report['user_profile_coverage'])
print('Precision@10 CI95:',report['precision_at_10_ci95_game_bootstrap'])
print('Limitations:')
for limitation in report['limitations']:print('-',limitation)


User profile coverage: 0.960200668896321
Precision@10 CI95: [0.04787541806020067, 0.051339464882943144]
Limitations:
- One observed pick is relevant per query; Precision@K=Recall@K/K with unfilled slots treated as misses.
- Retrospective query sees other nine picks, not real draft pick order or bans.
- This is validation evidence, not a fresh final-test claim or a winrate-uplift experiment.
- Observed picks reflect team/coach/patch constraints; frequency is implicit preference.
- Raw item attributes summarize past full-match performance, not champion mechanics.


,queries,candidate_coverage,recall_at_5,precision_at_5,mrr_at_5,recall_at_10,precision_at_10,mrr_at_10
content_based,2990.0,0.97893,0.298997,0.059799,0.138958,0.493645,0.049365,0.164486
popularity,2990.0,0.97893,0.465217,0.093043,0.221221,0.749164,0.074916,0.258210
player_frequency,2990.0,0.97893,0.483946,0.096789,0.240318,0.755853,0.075585,0.276346


,queries,candidate_coverage,recall_at_5,precision_at_5,mrr_at_5,recall_at_10,precision_at_10,mrr_at_10
content_based,2871.0,0.978753,0.311390,0.062278,0.144717,0.514107,0.051411,0.171304
popularity,2871.0,0.978753,0.465691,0.093138,0.221288,0.750261,0.075026,0.258430
player_frequency,2871.0,0.978753,0.485197,0.097039,0.241176,0.757227,0.075723,0.277318


## Kết luận minh chứng

Đã trình bày item profile, user profile có trọng số, cosine và đánh giá top-K theo Chương 6. Kết quả content-based hiện chưa vượt baseline phổ biến/tần suất cá nhân về truy hồi. Cần đánh giá liên quan thực tế và cải thiện trên validation khác trước khi tuyên bố lợi ích thực chiến; không sửa tham số để làm đẹp holdout này.